<a href="https://colab.research.google.com/github/Lisura123/J26-IT-348/blob/component2-tinyml/colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone --branch component2-tinyml https://github.com/Lisura123/J26-IT-348.git

fatal: destination path 'J26-IT-348' already exists and is not an empty directory.


In [3]:
%cd /content/J26-IT-348

!git branch --show-current
!git log -1 --oneline


/content/J26-IT-348
component2-tinyml
e1bf505 (HEAD -> component2-tinyml, origin/component2-tinyml) Created using Colab


In [4]:
# STEP 1b: Verify branch and commit
import os, subprocess

REPO_DIR = "/content/J26-IT-348"
EXPECTED_BRANCH = "component2-tinyml"
EXPECTED_COMMIT = "3c1f1a4"

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args],
                          capture_output=True, text=True, check=True).stdout.strip()

branch = git("branch", "--show-current")
commit = git("log", "-1", "--format=%h %s")
print("Branch :", branch)
print("Commit :", commit)
print("Branch OK :", branch == EXPECTED_BRANCH)
print("Commit OK :", commit.startswith(EXPECTED_COMMIT))
print("Component folder exists:", os.path.isdir(os.path.join(REPO_DIR, "component2-tinyml")))


Branch : component2-tinyml
Commit : e1bf505 Created using Colab
Branch OK : True
Commit OK : False
Component folder exists: True


In [5]:
# STEP 2: (Re)check repository, install dependencies, show versions
import os, sys, subprocess

REPO_URL = "https://github.com/Lisura123/J26-IT-348"
BRANCH = "component2-tinyml"
REPO_DIR = "/content/J26-IT-348"
COMPONENT_DIR = f"{REPO_DIR}/component2-tinyml"

if not os.path.isdir(REPO_DIR):
    print("Runtime was reset: cloning the repository again")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("Commit:", commit)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMPONENT_DIR}/requirements.txt"], check=True)

import numpy as np, pandas as pd, sklearn, pytest
print("Python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("pytest      :", pytest.__version__)
print("Install OK")

Commit: e1bf505 Created using Colab
Python      : 3.13.16
numpy       : 2.1.3
pandas      : 2.2.3
scikit-learn: 1.6.1
pytest      : 8.4.2
Install OK


In [6]:
# STEP 3: Import the repository modules (no formulas are copied into this notebook)
import sys
SRC_DIR = "/content/J26-IT-348/component2-tinyml/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from tinyml.contract import LABELS
from tinyml.validate import validate_dataset, file_sha256
from tinyml.splits import load_assignments, iter_folds
from tinyml.features import FEATURES, FEATURE_UNITS, FeatureConfig, fold_matrices, input_status
import tinyml.features

print("Modules loaded from:", tinyml.features.__file__)
print("Fault classes      :", LABELS)
print("Feature version    :", FeatureConfig().version)
print("Model input order  :")
for i, (f, u) in enumerate(zip(FEATURES, FEATURE_UNITS)):
    print(f"  {i}. {f} [{u}]")
print("Import OK")

Modules loaded from: /content/J26-IT-348/component2-tinyml/src/tinyml/features.py
Fault classes      : ('NORMAL', 'PARTIAL_SHADING', 'DUST', 'OPEN_CIRCUIT', 'SHORT_CIRCUIT')
Feature version    : pv8-v1
Model input order  :
  0. v_deficit [V]
  1. i_norm [mA/klx]
  2. p_norm [mW/klx]
  3. dust_vo_v [V]
  4. temp_spread [degC]
  5. temp_rise [degC]
  6. voltage_v [V]
  7. current_ma [mA]
Import OK


In [13]:
# STEP 4: Upload pv_fault_dataset.csv from your computer
import os
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.endswith(".csv")]
if not csv_files:
    raise SystemExit("No CSV uploaded. Run this cell again and choose pv_fault_dataset.csv")

DATA_PATH = os.path.abspath(csv_files[0])
print("Dataset file:", DATA_PATH)
print("Size (bytes):", os.path.getsize(DATA_PATH))

Saving pv_fault_dataset.csv to pv_fault_dataset (1).csv
Dataset file: /content/J26-IT-348/pv_fault_dataset (1).csv
Size (bytes): 957366


In [8]:
# STEP 5: Verify the dataset SHA-256 against the Phase 4 split configuration
import json

SPLITS_PATH = "/content/J26-IT-348/component2-tinyml/results/splits_runwise_v1.json"
with open(SPLITS_PATH, encoding="utf-8") as f:
    split_meta = json.load(f)["meta"]

expected_sha = split_meta["dataset_sha256"]
actual_sha = file_sha256(DATA_PATH)

print("Split version   :", split_meta["split_version"])
print("Expected SHA-256:", expected_sha)
print("Your file       :", actual_sha)
if actual_sha != expected_sha:
    raise SystemExit("STOP: this dataset does not match the one used for the Phase 4 splits.")
print("SHA-256 OK: dataset matches the split configuration.")

Split version   : runwise-v1
Expected SHA-256: 5fbb93471c056c274cbf81e8c317934e379b2484013147e4c2cacfa93421595b
Your file       : 5fbb93471c056c274cbf81e8c317934e379b2484013147e4c2cacfa93421595b
SHA-256 OK: dataset matches the split configuration.


In [9]:
# STEP 6: Dataset shape, class distribution and data integrity (read-only)
import pandas as pd

df = pd.read_csv(DATA_PATH)
print("Dataset shape (rows, columns):", df.shape)
print()

rows = pd.crosstab(df["label"], df["window"]).reindex(list(LABELS))
runs = pd.crosstab(df.drop_duplicates("run_id")["label"],
                   df.drop_duplicates("run_id")["window"]).reindex(list(LABELS))
print("Readings per class and window:")
print(rows.to_string())
print()
print("Runs per class and window:")
print(runs.to_string())
print()

report = validate_dataset(df)
print("Integrity check -> errors:", len(report["errors"]), "| warnings:", len(report["warnings"]))
for e in report["errors"]:
    print("  ERROR:", e)
for w in report["warnings"]:
    print("  -", w)

Dataset shape (rows, columns): (6000, 20)

Readings per class and window:
window           OPERATION  TRAINING
label                               
NORMAL                 600       600
PARTIAL_SHADING        600       600
DUST                   600       600
OPEN_CIRCUIT           600       600
SHORT_CIRCUIT          600       600

Runs per class and window:
window           OPERATION  TRAINING
label                               
NORMAL                  10        10
PARTIAL_SHADING         10        10
DUST                    10        10
OPEN_CIRCUIT            10        10
SHORT_CIRCUIT           10        10

Integrity check -> errors: 0 | warnings: 4
  - irradiance_lux at saturation (100000) in 76 rows
  - leakage: 'health_score' alone predicts the label with 69.5% run-wise accuracy; it must never be a model input
  - leakage: run_id contains the class code; use it only for grouping
  - sites missing classes (affects leave-one-site-out): {'SITE-B': ['PARTIAL_SHADING'], 'SITE-C': [

In [10]:
# STEP 7: Load the saved Phase 4 run-wise splits (no new splitting)
assignments, meta = load_assignments(SPLITS_PATH, df, dataset_sha256=actual_sha)
print("Split version:", meta["split_version"], "| seed:", meta["seed"],
      "| runs:", meta["runs"], "| made with scikit-learn", meta["sklearn_version"])
print()

split_check = {}
for protocol in ("cv", "temporal", "loso"):
    n_folds = 0
    for name, tr, te in iter_folds(df, assignments, protocol):
        shared = set(df.loc[tr, "run_id"]) & set(df.loc[te, "run_id"])
        assert not shared, f"{name}: runs in both train and test: {shared}"
        n_folds += 1
    split_check[protocol] = n_folds
    print(f"{protocol:9s}: {n_folds} folds, no run shared between train and test")

cv_runs = assignments[assignments["cv_fold"].astype(int) >= 0]
print()
print("CV uses only TRAINING runs:", set(cv_runs["window"]) == {"TRAINING"})
print("CV runs per fold and class:")
print(pd.crosstab(cv_runs["cv_fold"], cv_runs["label"]).reindex(columns=list(LABELS)).to_string())

Split version: runwise-v1 | seed: 42 | runs: 100 | made with scikit-learn 1.9.1

cv       : 5 folds, no run shared between train and test
temporal : 1 folds, no run shared between train and test
loso     : 6 folds, no run shared between train and test

CV uses only TRAINING runs: True
CV runs per fold and class:
label    NORMAL  PARTIAL_SHADING  DUST  OPEN_CIRCUIT  SHORT_CIRCUIT
cv_fold                                                            
0             2                2     2             2              2
1             2                2     2             2              2
2             2                2     2             2              2
3             2                2     2             2              2
4             2                2     2             2              2


In [11]:
# STEP 8: Feature smoke test on one CV fold (TRAINING window only, no model training)
import numpy as np

train_window = df[df["window"] == "TRAINING"]      # OPERATION is not used here
FOLD = "cv0"
fold_name, train_idx, val_idx = next((n, tr, te) for n, tr, te
                                     in iter_folds(train_window, assignments, "cv") if n == FOLD)

m = fold_matrices(train_window, train_idx, val_idx, fold=fold_name)   # fit on train runs only
cfg = m["config"]
Xtr, Xval = m["train"]["X"], m["test"]["X"]

print("Fold                  :", fold_name)
print("Training runs / rows  :", train_window.loc[train_idx, "run_id"].nunique(), "/", len(train_idx))
print("Validation runs / rows:", train_window.loc[val_idx, "run_id"].nunique(), "/", len(val_idx))
print("Training matrix shape :", Xtr.shape, Xtr.dtype)
print("Validation matrix shape:", Xval.shape, Xval.dtype)
print("Feature names         :", list(cfg.features))
print("V_ref (fitted on training runs):", {k: round(v, 4) for k, v in cfg.vref.items()})
print("Fitted on             :", cfg.fitted_on)
print()

for side, title in (("train", "Training part"), ("test", "Validation part")):
    status = m[side]["status"]
    table = pd.crosstab(train_window.loc[status.index, "label"], status).reindex(list(LABELS)).fillna(0).astype(int)
    print(f"Quality gates, {title} (readings per class):")
    print(table.to_string())
    print()

checks = {
    "8 features in agreed order": list(cfg.features) == list(FEATURES),
    "training inputs finite": bool(np.isfinite(Xtr).all()),
    "validation inputs finite": bool(np.isfinite(Xval).all()),
    "inputs are float32": Xtr.dtype == np.float32 and Xval.dtype == np.float32,
    "fitted on training rows only": cfg.fitted_on["train_rows"] == len(train_idx),
    "no run in both train and validation": not (set(train_window.loc[train_idx, "run_id"])
                                                & set(train_window.loc[val_idx, "run_id"])),
    "OPERATION rows not used": not (df.loc[list(train_idx) + list(val_idx), "window"] == "OPERATION").any(),
    "training features scaled (mean~0, std~1)": bool(np.allclose(Xtr.mean(0), 0, atol=1e-3)
                                                     and np.allclose(Xtr.std(0), 1, atol=1e-2)),
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

Fold                  : cv0
Training runs / rows  : 40 / 2400
Validation runs / rows: 10 / 600
Training matrix shape : (2324, 8) float32
Validation matrix shape: (600, 8) float32
Feature names         : ['v_deficit', 'i_norm', 'p_norm', 'dust_vo_v', 'temp_spread', 'temp_rise', 'voltage_v', 'current_ma']
V_ref (fitted on training runs): {'a': 2.5645, 'b': 0.0711, 'c': -0.0095}
Fitted on             : {'fold': 'cv0', 'train_rows': 2400, 'ok_rows': 2324, 'normal_rows_for_vref': 480}

Quality gates, Training part (readings per class):
input_status     LUX_SATURATED   OK
label                              
NORMAL                       0  480
PARTIAL_SHADING             27  453
DUST                        49  431
OPEN_CIRCUIT                 0  480
SHORT_CIRCUIT                0  480

Quality gates, Validation part (readings per class):
input_status      OK
label               
NORMAL           120
PARTIAL_SHADING  120
DUST             120
OPEN_CIRCUIT     120
SHORT_CIRCUIT    120

PASS  8 f

In [12]:
# STEP 9: Run the repository tests and save a summary of this Colab check
import json, subprocess, sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
                        cwd=COMPONENT_DIR, capture_output=True, text=True)
tests_line = result.stdout.strip().splitlines()[-1] if result.stdout.strip() else result.stderr[-300:]
print(result.stdout[-1500:])
print("Test result:", tests_line)

summary = {
    "commit": commit,
    "python": sys.version.split()[0],
    "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "dataset_sha256_ok": actual_sha == expected_sha,
    "dataset_shape": list(df.shape),
    "validation_errors": len(report["errors"]),
    "validation_warnings": len(report["warnings"]),
    "health_score_runwise_accuracy": report["leakage"]["single_column_runwise_accuracy"],
    "split_folds": split_check,
    "fold": fold_name,
    "train_matrix_shape": list(Xtr.shape),
    "validation_matrix_shape": list(Xval.shape),
    "vref": cfg.vref,
    "feature_checks": {k: bool(v) for k, v in checks.items()},
    "tests": tests_line,
}
with open("/content/colab_env_check.json", "w") as f:
    json.dump(summary, f, indent=1)
print(json.dumps(summary, indent=1))

from google.colab import files
files.download("/content/colab_env_check.json")

hout_con1/clone'), 'other': '0b453b10c39c6f62d844f7945f8f2528f73ff9f7'}

    def test_step5_refuses_without_confirmation(repos):
        calls = []
>       with pytest.raises(SystemExit, match="CONFIRM_QUANTIZATION = True"):
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E       Failed: DID NOT RAISE <class 'SystemExit'>

tests/test_notebooks.py:117: Failed
----------------------------- Captured stdout call -----------------------------
x                            PASS
INT8 accepted: True | -
Host C tests : ALL PASSED
=========================== short test summary info ============================
FAILED tests/test_notebook_pruning.py::test_seven_steps_guards_and_training_only
FAILED tests/test_notebook_pruning.py::test_step1_refuses_missing_or_malformed_hash
FAILED tests/test_notebook_pruning.py::test_step1_checks_out_exactly_the_approved_older_revision
FAILED tests/test_notebook_pruning.py::test_step1_refuses_commit_from_another_branch
FAILED tests/test_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>